# M512 Hotel Booking Demand — Forensic EDA v1

**Purpose:** exploratory evidence base. This notebook audits structure, data quality and broad relationships before any explanatory story is selected.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

CSV=Path("hotel_bookings.csv")
if not CSV.exists():
    import kagglehub
    folder=Path(kagglehub.dataset_download("jessemostipak/hotel-booking-demand"))
    CSV=next(iter(folder.rglob("hotel_bookings.csv")))
df=pd.read_csv(CSV)
print(df.shape)
display(df.head())

## 1. Structural audit

In [ ]:
audit=pd.DataFrame({
"dtype":df.dtypes.astype(str),
"missing_n":df.isna().sum(),
"missing_pct":(df.isna().mean()*100).round(2),
"n_unique":df.nunique(dropna=False)
}).sort_values("missing_pct",ascending=False)
print("Exact duplicate-looking rows:",int(df.duplicated().sum()))
display(audit)

Exact duplicate-looking rows are not automatically removed because the source has no unique booking identifier. Identical anonymised rows can represent separate bookings.

## 2. Dates and derived fields

In [ ]:
month_num={"January":1,"February":2,"March":3,"April":4,"May":5,"June":6,"July":7,"August":8,"September":9,"October":10,"November":11,"December":12}
df["arrival_month_num"]=df.arrival_date_month.map(month_num)
df["arrival_date"]=pd.to_datetime(dict(year=df.arrival_date_year,month=df.arrival_month_num,day=df.arrival_date_day_of_month))
df["stay_nights"]=df.stays_in_weekend_nights+df.stays_in_week_nights
df["guests"]=df.adults+df.children.fillna(0)+df.babies
df["has_prior_success"]=df.previous_bookings_not_canceled.gt(0)
df["has_special_request"]=df.total_of_special_requests.gt(0)
bins=[-1,7,30,90,180,365,np.inf]; labels=["0–7","8–30","31–90","91–180","181–365","366+"]
df["lead_band"]=pd.cut(df.lead_time,bins,labels=labels,ordered=True)
print(df.arrival_date.min(),df.arrival_date.max())

## 3. Quality flags and outcome leakage

In [ ]:
quality=pd.Series({
"zero_guests":(df.guests==0).sum(),
"zero_stay_nights":(df.stay_nights==0).sum(),
"negative_adr":(df.adr<0).sum(),
"adr_gt_1000":(df.adr>1000).sum(),
"missing_children":df.children.isna().sum(),
"missing_country":df.country.isna().sum()
},name="count")
display(quality.to_frame())
display((pd.crosstab(df.reservation_status,df.is_canceled,normalize="index")*100).round(1))

reservation_status and reservation_status_date encode the realised outcome and must not be treated as cancellation drivers.

## 4. Systematic one-way cancellation screen

In [ ]:
candidate_cols=["hotel","lead_band","market_segment","distribution_channel","customer_type","deposit_type","is_repeated_guest","has_prior_success","has_special_request","total_of_special_requests","required_car_parking_spaces","meal","reserved_room_type","arrival_date_month","arrival_date_year"]

def rate_table(col):
    g=df.groupby(col,dropna=False,observed=False).is_canceled.agg(["size","sum","mean"]).reset_index()
    g["cancel_rate_pct"]=(g["mean"]*100).round(1)
    return g.sort_values("size",ascending=False).drop(columns="mean")

for col in candidate_cols:
    print("\n###",col)
    display(rate_table(col).head(15))

## 5. Lead-time and segment robustness

In [ ]:
display((df.groupby("lead_band",observed=True).is_canceled.agg(["size","mean"]).assign(mean=lambda x:x["mean"]*100)).round(1))
display((df.pivot_table(index="lead_band",columns="hotel",values="is_canceled",aggfunc="mean",observed=True)*100).round(1))
display((df.pivot_table(index="lead_band",columns="arrival_date_year",values="is_canceled",aggfunc="mean",observed=True)*100).round(1))

seg=df.groupby("market_segment").is_canceled.agg(bookings="size",cancellations="sum",rate="mean")
seg["rate"]*=100
seg["share_cancel_pct"]=seg.cancellations/df.is_canceled.sum()*100
display(seg.sort_values("cancellations",ascending=False).round(1))

## 6. Interaction screen

In [ ]:
interaction=(df.groupby(["market_segment","lead_band"],observed=True).is_canceled
             .agg(bookings="size",cancellations="sum",rate="mean").reset_index())
interaction["rate"]*=100
display(interaction[interaction.bookings>=100].sort_values("cancellations",ascending=False).head(25).round(1))

triple=(df.groupby(["hotel","market_segment","lead_band"],observed=True).is_canceled
        .agg(bookings="size",cancellations="sum",rate="mean").reset_index())
triple["rate"]*=100
display(triple[triple.bookings>=100].sort_values("cancellations",ascending=False).head(30).round(1))

## 7. v1 evidence gate

KEEP: cancellation burden, lead-time gradient, market-segment concentration, hotel context, recorded prior successful-booking history.

RESERVE: special requests and distribution channel.

QUALITY ONLY: deposit type and parking-space anomaly.

REJECT AS HEADLINE: booking changes, previous cancellations and ADR bands until deeper robustness checks are complete.